# Aula 2 — Parte 2: demonstração

Execute as células na ordem apresentada.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEMENTE = 42
pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 10)


def titulo(n, texto):
    print()
    print("=" * 70)
    print(f"BLOCO {n} - {texto}")
    print("=" * 70)


## Bloco 1


In [5]:
titulo(1, "CARREGAR E MONTAR O DICIONARIO DE DADOS")

wine = load_wine(as_frame=True)
df = wine.frame                 # ja vem com a coluna "target"

print("formato do DataFrame:", df.shape, "-> 178 amostras, 13 atributos + rotulo")
print("\ntipos de cada coluna:")
print(df.dtypes)

X = df.drop(columns="target")   # matriz de atributos
y = df["target"]                # vetor de rotulos

dicionario = pd.DataFrame({
    "atributo": X.columns,
    "tipo": [str(t) for t in X.dtypes],
    "exemplo": [X[c].iloc[0] for c in X.columns],
})
print("\nDicionario de dados:")
print(dicionario.to_string(index=False))

print("""
LEIA COM ATENCAO: 13 atributos quimicos, todos numericos, mas em escalas
bem diferentes. Compare a coluna 'magnesium' (dezenas) com 'proline'
(centenas/milhares) no describe() abaixo.""")
print("\nEstatisticas descritivas (min e max de cada coluna):")
print(X.describe().T[["min", "max"]].round(2))



BLOCO 1 - CARREGAR E MONTAR O DICIONARIO DE DADOS
formato do DataFrame: (178, 14) -> 178 amostras, 13 atributos + rotulo

tipos de cada coluna:
alcohol                         float64
malic_acid                      float64
ash                             float64
alcalinity_of_ash               float64
magnesium                       float64
total_phenols                   float64
flavanoids                      float64
nonflavanoid_phenols            float64
proanthocyanins                 float64
color_intensity                 float64
hue                             float64
od280/od315_of_diluted_wines    float64
proline                         float64
target                            int64
dtype: object

Dicionario de dados:
                    atributo    tipo  exemplo
                     alcohol float64    14.23
                  malic_acid float64     1.71
                         ash float64     2.43
           alcalinity_of_ash float64    15.60
                   magnesium 

## Bloco 2


In [7]:
titulo(2, "FORMULANDO X E y EXPLICITAMENTE")

print(f"unidade de analise : uma amostra de vinho (uma linha)")
print(f"X.shape = {X.shape}  -> {X.shape[0]} amostras, {X.shape[1]} atributos")
print(f"y.shape = {y.shape}  -> {y.shape[0]} rotulos")
print("\nclasses (cultivares) e contagem:")
print(y.value_counts().sort_index())

print("""
E supervisionado: y (o cultivar) e conhecido para as 178 amostras. Note que
o conjunto NAO e perfeitamente balanceado (59 / 71 / 48) - diferente do
Iris, que tinha exatamente 50 de cada classe. Isso importa para a escolha
de metricas.""")



BLOCO 2 - FORMULANDO X E y EXPLICITAMENTE
unidade de analise : uma amostra de vinho (uma linha)
X.shape = (178, 13)  -> 178 amostras, 13 atributos
y.shape = (178,)  -> 178 rotulos

classes (cultivares) e contagem:
target
0    59
1    71
2    48
Name: count, dtype: int64

E supervisionado: y (o cultivar) e conhecido para as 178 amostras. Note que
o conjunto NAO e perfeitamente balanceado (59 / 71 / 48) - diferente do
Iris, que tinha exatamente 50 de cada classe. Isso importa para a escolha
de metricas.


## Bloco 3


In [10]:
titulo(3, "REFORCO: K-NN COM E SEM PADRONIZACAO")

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.30, random_state=SEMENTE, stratify=y,
)
print(f"treino: {X_treino.shape[0]} amostras | teste: {X_teste.shape[0]} amostras")

sem_padronizacao = KNeighborsClassifier(n_neighbors=5)
sem_padronizacao.fit(X_treino, y_treino)
ac_sem = accuracy_score(y_teste, sem_padronizacao.predict(X_teste))

com_padronizacao = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
com_padronizacao.fit(X_treino, y_treino)
ac_com = accuracy_score(y_teste, com_padronizacao.predict(X_teste))


print(f"\nsem padronizacao: {ac_sem:.4f}")
print(f"com padronizacao: {ac_com:.4f}")

print(f"""
LEITURA: no Iris a padronizacao quase nao mudava o resultado, porque as
quatro colunas ja estavam na mesma unidade (cm). Aqui a diferenca costuma
ser grande ({ac_sem:.1%} -> {ac_com:.1%}) porque 'proline' (na casa das
centenas/milhares) domina a distancia euclidiana diante de colunas como
'hue' (na casa das unidades). A licao da Parte 1 deixa de ser hipotetica:
no Wine, padronizar e essencial.""")



BLOCO 3 - REFORCO: K-NN COM E SEM PADRONIZACAO
treino: 124 amostras | teste: 54 amostras
     alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  ...  proanthocyanins  \
12     13.75        1.73  2.41               16.0       89.0  ...             1.81   
30     13.73        1.50  2.70               22.5      101.0  ...             2.38   
36     13.28        1.64  2.84               15.5      110.0  ...             1.36   
31     13.58        1.66  2.36               19.1      106.0  ...             1.95   
120    11.45        2.40  2.42               20.0       96.0  ...             1.83   
..       ...         ...   ...                ...        ...  ...              ...   
168    13.58        2.58  2.69               24.5      105.0  ...             1.54   
114    12.08        1.39  2.50               22.5       84.0  ...             1.04   
152    13.11        1.90  2.75               25.5      116.0  ...             1.56   
136    12.25        4.72  2.54               21.0 

## Bloco 4


In [9]:
titulo(4, "COMPARANDO IRIS E WINE NA MESMA NOTACAO")

print("""
  Elemento              | Iris                    | Wine
  ------------------------------------------------------------------
  Unidade de analise     | uma flor medida         | uma amostra de vinho
  n (exemplos)           | 150                     | 178
  d (atributos)          | 4, mesma unidade (cm)   | 13, unidades diferentes
  y (rotulo)             | especie (3 classes)     | cultivar (3 classes)
  Padronizacao           | quase nao muda          | essencial

FIM DA DEMONSTRACAO.
Abra agora o lab02_parte2_exercicios.py e resolva os TODO.""")



BLOCO 4 - COMPARANDO IRIS E WINE NA MESMA NOTACAO

  Elemento              | Iris                    | Wine
  ------------------------------------------------------------------
  Unidade de analise     | uma flor medida         | uma amostra de vinho
  n (exemplos)           | 150                     | 178
  d (atributos)          | 4, mesma unidade (cm)   | 13, unidades diferentes
  y (rotulo)             | especie (3 classes)     | cultivar (3 classes)
  Padronizacao           | quase nao muda          | essencial

FIM DA DEMONSTRACAO.
Abra agora o lab02_parte2_exercicios.py e resolva os TODO (30 min).
